John S. Anvik, 2026  
University of Lethbridge Student ID: 001224816

In [1]:
library(tidyverse)
library(tidymodels)
library(rpart.plot) # for plotting decision tree
library(vip) # for ranking feature importance
library(ranger) # engine for random forest

# set random seed
set.seed(42)

Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'tibble' was built under R version 4.5.3"
Warning message:
"package 'tidyr' was built under R version 4.5.3"
Warning message:
"package 'readr' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
Warning message:
"package 'dplyr' was built under R version 4.5.3"
Warning message:
"package 'lubridate' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<

In [2]:

data_list <- list()

# TO RUN MODEL ON ALL 10 FOCAL MALES:
folder_path <- "C:\\Users\\johna\\OneDrive\\Documents\\Birdsong\\data\\by_id"
files = list.files(folder_path, full.names=TRUE)

for (file in files) {
    data_list <- append(data_list, list(read_csv(file)))
}

Rows: 60648 Columns: 644
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (3): ID, attention_target_id, event_id
dbl  (640): UST, time_rel_sunrise, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_...
dttm   (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2166 Columns: 360
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (2): ID, attention_target_id
dbl  (357): UST, time_rel_sunrise, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_...
dttm   (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 9804 Columns: 437
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr    (3): ID, attention_target_id, event_id

In [3]:
focal_ids <- list()
for (i in seq_along(data_list)) {
    id <- data_list[[i]] %>% pull(ID) %>% unique()
    # print(id)
    focal_ids[[id]] <- id
}
# focal_ids <- list("YKY")
names(data_list) <- focal_ids
# names(data_list) <- "YKY"

In [4]:
# In full dataset, how many of each song type for the focal male?

MODEL_TYPE <- "Baseline Classifier"

accuracy <- tibble()

for (id in focal_ids) {    n_ust <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate( ID=id, estimated_accuracy = n/nrow(data_list[[id]]),model=MODEL_TYPE, model_type="Baseline") %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)

# A tibble: 10 × 3
   ID     estimated_accuracy model              
   <chr>               <dbl> <chr>              
 1 DgTbO               0.184 Baseline Classifier
 2 KLgK                0.184 Baseline Classifier
 3 KRDg                0.149 Baseline Classifier
 4 LbOW                0.149 Baseline Classifier
 5 RbDgY               0.132 Baseline Classifier
 6 RRO                 0.132 Baseline Classifier
 7 TbTbLg              0.163 Baseline Classifier
 8 TbWY                0.163 Baseline Classifier
 9 WLpTb               0.142 Baseline Classifier
10 YKY                 0.142 Baseline Classifier
[1] "data/results/Baseline Classifier_test_accuracies.csv"


In [5]:
MODEL_TYPE <- "N greater than 30 Baseline Classifier"
min_n <- 30

accuracy <- tibble()

for (id in focal_ids) {
    ust_counts <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n())
    top_n_ust <- ust_counts %>% filter(n > min_n) %>% pull(UST)
    # print(top_n_ust)
    n_ust <- data_list[[id]] %>% filter(UST %in% top_n_ust) %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate(ID = id, estimated_accuracy = n/nrow(data_list[[id]] %>% filter(UST %in% top_n_ust)),model=MODEL_TYPE, model_type="Baseline") %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)

# A tibble: 10 × 3
   ID     estimated_accuracy model                                
   <chr>               <dbl> <chr>                                
 1 DgTbO               0.185 N greater than 30 Baseline Classifier
 2 KLgK                0.19  N greater than 30 Baseline Classifier
 3 KRDg                0.149 N greater than 30 Baseline Classifier
 4 LbOW                0.256 N greater than 30 Baseline Classifier
 5 RbDgY               0.133 N greater than 30 Baseline Classifier
 6 RRO                 0.159 N greater than 30 Baseline Classifier
 7 TbTbLg              0.164 N greater than 30 Baseline Classifier
 8 TbWY                0.170 N greater than 30 Baseline Classifier
 9 WLpTb               0.142 N greater than 30 Baseline Classifier
10 YKY                 0.182 N greater than 30 Baseline Classifier
[1] "data/results/N greater than 30 Baseline Classifier_test_accuracies.csv"


In [6]:
MODEL_TYPE <- "Top 5 Classes Baseline Classifier"
N <- 5

accuracy <- tibble()

for (id in focal_ids) {
    ust_counts <- data_list[[id]] %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        arrange(desc(n))
    top_n_ust <- ust_counts %>% slice(0:5) %>% pull(UST)
    # print(top_n_ust)
    n_ust <- data_list[[id]] %>% filter(UST %in% top_n_ust) %>%
        group_by(UST) %>%
        summarize(n = n()) %>%
        mutate(ID = id, estimated_accuracy = n/nrow(data_list[[id]] %>% filter(UST %in% top_n_ust)),model=MODEL_TYPE, model_type="Baseline") %>%
        arrange(desc(estimated_accuracy)) %>%
        rename(Top_UST = UST) %>%
        slice(1)
    accuracy <- accuracy %>% bind_rows(n_ust)
    # print(n_ust)
}
accuracy <- accuracy %>% select(ID, estimated_accuracy, model)
print(accuracy)

filename <- paste0("data/results/", MODEL_TYPE, "_test_accuracies.csv")
print(filename)
write_csv(accuracy, filename)


# A tibble: 10 × 3
   ID     estimated_accuracy model                            
   <chr>               <dbl> <chr>                            
 1 DgTbO               0.291 Top 5 Classes Baseline Classifier
 2 KLgK                0.291 Top 5 Classes Baseline Classifier
 3 KRDg                0.291 Top 5 Classes Baseline Classifier
 4 LbOW                0.291 Top 5 Classes Baseline Classifier
 5 RbDgY               0.288 Top 5 Classes Baseline Classifier
 6 RRO                 0.288 Top 5 Classes Baseline Classifier
 7 TbTbLg              0.237 Top 5 Classes Baseline Classifier
 8 TbWY                0.237 Top 5 Classes Baseline Classifier
 9 WLpTb               0.276 Top 5 Classes Baseline Classifier
10 YKY                 0.276 Top 5 Classes Baseline Classifier
[1] "data/results/Top 5 Classes Baseline Classifier_test_accuracies.csv"


In [7]:
top_n_ust

[1] 21  8  3  9 19